In [ ]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
main_df = pd.read_csv('/content/drive/MyDrive/Datasets/df_temp_cascade_multi_apps.csv') # combined_apps_reviews # gen_ai_reviews_main_dataset #gen_ai_reviews_5_apps_dataset #PlayStoreReviewsMultipleCategoires

main_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500135 entries, 0 to 500134
Data columns (total 11 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   app_id                  500135 non-null  object 
 1   app_name                500135 non-null  object 
 2   app_category            500135 non-null  object 
 3   app_aggregate_rating    500135 non-null  float64
 4   review_text             500117 non-null  object 
 5   review_rating           500135 non-null  int64  
 6   helpful_votes           500135 non-null  int64  
 7   review_date             500135 non-null  object 
 8   scraped_at              500135 non-null  object 
 9   review_rating_original  500135 non-null  int64  
 10  sentiment               500135 non-null  float64
dtypes: float64(2), int64(3), object(6)
memory usage: 42.0+ MB


In [ ]:
sample_percentage = 1 # Set the desired sample percentage (e.g., 0.50 for 50%, 1 for 100%)
df = main_df.sample(frac=sample_percentage, random_state=42) # random_state for reproducibility

print(f"New DataFrame size: {len(df)} records ({sample_percentage*100}% of original)")


New DataFrame size: 500135 records (100% of original)


In [ ]:
import numpy as np
import pandas as pd

analysis_df = df[["review_text", "review_rating_original", "review_rating", "sentiment"]].copy()
numeric_columns = ["review_rating_original", "review_rating", "sentiment"]

for column in numeric_columns:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors="coerce")

analysis_df = analysis_df.dropna(subset=numeric_columns)
analysis_df = analysis_df[
    analysis_df["review_rating_original"].isin([1, 2, 3, 4, 5])
    & analysis_df["review_rating"].isin([1, 2, 3, 4, 5])
    & analysis_df["sentiment"].between(-1, 1)
].copy()

group_names = ["Negative", "Neutral", "Positive"]

analysis_df["sentiment_category"] = np.select(
    [analysis_df["sentiment"] <= -0.05, analysis_df["sentiment"] >= 0.05],
    ["Negative", "Positive"],
    default="Neutral"
)

analysis_df["original_category"] = pd.cut(
    analysis_df["review_rating_original"], bins=[0, 2, 3, 5], labels=group_names
).astype(str)

analysis_df["proposed_category"] = pd.cut(
    analysis_df["review_rating"], bins=[0, 2, 3, 5], labels=group_names
).astype(str)

misaligned_reviews = analysis_df[
    analysis_df["original_category"] != analysis_df["sentiment_category"]
].copy()

misaligned_reviews["corrected"] = misaligned_reviews["proposed_category"] == misaligned_reviews["sentiment_category"]
misaligned_reviews["outcome"] = np.where(misaligned_reviews["corrected"], "Corrected", "Still misaligned")

original_counts = analysis_df["original_category"].value_counts().reindex(group_names, fill_value=0)
misaligned_counts = misaligned_reviews["original_category"].value_counts().reindex(group_names, fill_value=0)
corrected_counts = misaligned_reviews.loc[misaligned_reviews["corrected"], "original_category"].value_counts().reindex(group_names, fill_value=0)

results = pd.DataFrame({
    "Total original reviews": original_counts,
    "Originally misaligned": misaligned_counts,
    "Corrected by proposed labels": corrected_counts
})

results["Still misaligned"] = results["Originally misaligned"] - results["Corrected by proposed labels"]
results.loc["Overall"] = results.sum()

results.insert(2, "Originally misaligned (%)", np.where(
    results["Total original reviews"] > 0,
    100 * results["Originally misaligned"] / results["Total original reviews"],
    np.nan
))

results["Correctly reassigned (%)"] = np.where(
    results["Originally misaligned"] > 0,
    100 * results["Corrected by proposed labels"] / results["Originally misaligned"],
    np.nan
)

results.index.name = "Original rating category"

transition_table = pd.crosstab(
    [misaligned_reviews["original_category"], misaligned_reviews["sentiment_category"]],
    misaligned_reviews["proposed_category"]
).reindex(columns=group_names, fill_value=0)

print(f"Total reviews evaluated: {len(analysis_df):,}")
print(f"Originally misaligned reviews: {len(misaligned_reviews):,}")
print(f"Originally misaligned reviews (%): {100 * len(misaligned_reviews) / len(analysis_df):.2f}%")
print(f"Corrected reviews: {misaligned_reviews['corrected'].sum():,}")
print(f"Still misaligned reviews: {(~misaligned_reviews['corrected']).sum():,}")
print(f"Percentage of misaligned reviews correctly reassigned: {results.loc['Overall', 'Correctly reassigned (%)']:.2f}%")

print("\nMisalignment correction comparison:")
display(results.round(2))

print("\nReassignment of originally misaligned reviews:")
display(transition_table)

Total reviews evaluated: 500,135
Originally misaligned reviews: 155,489
Originally misaligned reviews (%): 31.09%
Corrected reviews: 45,777
Still misaligned reviews: 109,712
Percentage of misaligned reviews correctly reassigned: 29.44%

Misalignment correction comparison:


,Total original reviews,Originally misaligned,Originally misaligned (%),Corrected by proposed labels,Still misaligned,Correctly reassigned (%)
Original rating category,,,,,,
Negative,147003,77028,52.40,18013,59015,23.39
Neutral,31407,25690,81.80,16433,9257,63.97
Positive,321725,52771,16.40,11331,41440,21.47
Overall,500135,155489,31.09,45777,109712,29.44



Reassignment of originally misaligned reviews:


proposed_category                     Negative  Neutral  Positive
original_category sentiment_category                             
Negative          Neutral                32486       29         0
                  Positive               23405     3124     17984
Neutral           Negative                8191      158         1
                  Positive                6137     2961      8242
Positive          Negative               11112      342         5
                  Neutral                41079      219        14

In [ ]:
df['app_category'].value_counts()

,count
app_category,
entertainment,100187
education,100023
ai,100003
music,100000
health and fitness,99922


In [ ]:
df['app_name'].value_counts()

,count
app_name,
Angel: Watch Movies & TV Shows,10000
YouTube Music,10000
Apple Music,10000
Audiomack: Music Downloader,10000
"Character AI: Chat, Talk, Text",10000
Calorie Counter - MyNetDiary,10000
Shazam: Find Music & Concerts,10000
Amazon Music: Songs & Podcasts,10000
AI Chatbot - Nova,10000


In [ ]:
import numpy as np
import pandas as pd

app_names = ["ChatGPT", "Netflix", "Udemy - Online Courses", "MyFitnessPal: Calorie Counter", "SoundCloud: The Music You Love "]

candidate_columns = [
    "app_name", "app", "App Name", "App", "appName",
    "app_title", "application_name", "application", "name", "title"
]

app_column = next(
    (column for column in candidate_columns
     if column in df.columns
     and df[column].astype("string").str.strip().isin([name.strip() for name in app_names]).any()),
    None
)

if app_column is None:
    raise ValueError("Application name column not found. Check df.columns.")

app_subsets = {
    name: df[df[app_column].astype("string").str.strip() == name.strip()].copy()
    for name in app_names
}

group_names = ["Negative", "Neutral", "Positive"]

for app_name, app_df in app_subsets.items():
    print(f"\n{'=' * 80}")
    print(f"Application: {app_name}")
    print(f"{'=' * 80}")

    analysis_df = app_df[["review_text", "review_rating_original", "review_rating", "sentiment"]].copy()
    numeric_columns = ["review_rating_original", "review_rating", "sentiment"]

    for column in numeric_columns:
        analysis_df[column] = pd.to_numeric(analysis_df[column], errors="coerce")

    analysis_df = analysis_df.dropna(subset=numeric_columns)
    analysis_df = analysis_df[
        analysis_df["review_rating_original"].isin([1, 2, 3, 4, 5])
        & analysis_df["review_rating"].isin([1, 2, 3, 4, 5])
        & analysis_df["sentiment"].between(-1, 1)
    ].copy()

    if analysis_df.empty:
        print("No valid reviews found.")
        continue

    analysis_df["sentiment_category"] = np.select(
        [analysis_df["sentiment"] <= -0.05, analysis_df["sentiment"] >= 0.05],
        ["Negative", "Positive"], default="Neutral"
    )

    analysis_df["original_category"] = pd.cut(
        analysis_df["review_rating_original"], bins=[0, 2, 3, 5], labels=group_names
    ).astype(str)

    analysis_df["proposed_category"] = pd.cut(
        analysis_df["review_rating"], bins=[0, 2, 3, 5], labels=group_names
    ).astype(str)

    misaligned_reviews = analysis_df[
        analysis_df["original_category"] != analysis_df["sentiment_category"]
    ].copy()

    misaligned_reviews["corrected"] = (
        misaligned_reviews["proposed_category"] == misaligned_reviews["sentiment_category"]
    )

    misaligned_reviews["outcome"] = np.where(
        misaligned_reviews["corrected"], "Corrected", "Still misaligned"
    )

    original_counts = analysis_df["original_category"].value_counts().reindex(group_names, fill_value=0)
    misaligned_counts = misaligned_reviews["original_category"].value_counts().reindex(group_names, fill_value=0)
    corrected_counts = misaligned_reviews.loc[
        misaligned_reviews["corrected"], "original_category"
    ].value_counts().reindex(group_names, fill_value=0)

    results = pd.DataFrame({
        "Total original reviews": original_counts,
        "Originally misaligned": misaligned_counts,
        "Corrected by proposed labels": corrected_counts
    })

    results["Still misaligned"] = results["Originally misaligned"] - results["Corrected by proposed labels"]
    results.loc["Overall"] = results.sum()

    results.insert(2, "Originally misaligned (%)", np.where(
        results["Total original reviews"] > 0,
        100 * results["Originally misaligned"] / results["Total original reviews"],
        np.nan
    ))

    results["Correctly reassigned (%)"] = np.where(
        results["Originally misaligned"] > 0,
        100 * results["Corrected by proposed labels"] / results["Originally misaligned"],
        np.nan
    )

    results.index.name = "Original rating category"

    transition_table = pd.crosstab(
        [misaligned_reviews["original_category"], misaligned_reviews["sentiment_category"]],
        misaligned_reviews["proposed_category"]
    ).reindex(columns=group_names, fill_value=0)

    print(f"Total reviews evaluated: {len(analysis_df):,}")
    print(f"Originally misaligned reviews: {len(misaligned_reviews):,}")
    print(f"Originally misaligned reviews (%): {100 * len(misaligned_reviews) / len(analysis_df):.2f}%")
    print(f"Corrected reviews: {misaligned_reviews['corrected'].sum():,}")
    print(f"Still misaligned reviews: {(~misaligned_reviews['corrected']).sum():,}")
    print(f"Percentage of misaligned reviews correctly reassigned: {results.loc['Overall', 'Correctly reassigned (%)']:.2f}%")

    print("\nMisalignment correction comparison:")
    display(results.round(2))

    print("\nReassignment of originally misaligned reviews:")
    display(transition_table)


Application: ChatGPT
Total reviews evaluated: 10,000
Originally misaligned reviews: 2,456
Originally misaligned reviews (%): 24.56%
Corrected reviews: 269
Still misaligned reviews: 2,187
Percentage of misaligned reviews correctly reassigned: 10.95%

Misalignment correction comparison:


,Total original reviews,Originally misaligned,Originally misaligned (%),Corrected by proposed labels,Still misaligned,Correctly reassigned (%)
Original rating category,,,,,,
Negative,720,532,73.89,71,461,13.35
Neutral,406,305,75.12,104,201,34.10
Positive,8874,1619,18.24,94,1525,5.81
Overall,10000,2456,24.56,269,2187,10.95



Reassignment of originally misaligned reviews:


proposed_category                     Negative  Neutral  Positive
original_category sentiment_category                             
Negative          Neutral                  220        0         0
                  Positive                  52      189        71
Neutral           Negative                  27        2         0
                  Positive                  25      174        77
Positive          Negative                  86        5         0
                  Neutral                 1520        8         0


Application: Netflix
Total reviews evaluated: 10,000
Originally misaligned reviews: 3,598
Originally misaligned reviews (%): 35.98%
Corrected reviews: 794
Still misaligned reviews: 2,804
Percentage of misaligned reviews correctly reassigned: 22.07%

Misalignment correction comparison:


,Total original reviews,Originally misaligned,Originally misaligned (%),Corrected by proposed labels,Still misaligned,Correctly reassigned (%)
Original rating category,,,,,,
Negative,4239,2153,50.79,343,1810,15.93
Neutral,524,396,75.57,248,148,62.63
Positive,5237,1049,20.03,203,846,19.35
Overall,10000,3598,35.98,794,2804,22.07


... (10 outputs left out)


Application: SoundCloud: The Music You Love 
Total reviews evaluated: 10,000
Originally misaligned reviews: 3,482
Originally misaligned reviews (%): 34.82%
Corrected reviews: 807
Still misaligned reviews: 2,675
Percentage of misaligned reviews correctly reassigned: 23.18%

Misalignment correction comparison:


,Total original reviews,Originally misaligned,Originally misaligned (%),Corrected by proposed labels,Still misaligned,Correctly reassigned (%)
Original rating category,,,,,,
Negative,1881,1117,59.38,227,890,20.32
Neutral,594,477,80.30,267,210,55.97
Positive,7525,1888,25.09,313,1575,16.58
Overall,10000,3482,34.82,807,2675,23.18



Reassignment of originally misaligned reviews:


proposed_category                     Negative  Neutral  Positive
original_category sentiment_category                             
Negative          Neutral                  507        1         0
                  Positive                 292       91       226
Neutral           Negative                 148        3         0
                  Positive                 113       94       119
Positive          Negative                 306        9         0
                  Neutral                 1565        7         1

In [ ]:
# import re
# import pandas as pd
# from nltk.stem import PorterStemmer

# search_terms = [
#     'generate image', 'generate picture', 'generate photo', 'generate figure', 'generate diagram', 'generate flowchart', 'generate illustration', 'generate graphic', 'generate artwork',
#     'create image', 'create picture', 'create photo', 'create figure', 'create diagram', 'create flowchart', 'create illustration', 'create graphic', 'create artwork',
#     'draw image', 'draw picture', 'draw photo', 'draw figure', 'draw diagram', 'draw flowchart', 'draw illustration', 'draw graphic', 'draw artwork',
#     'make image', 'make picture', 'make photo', 'make figure', 'make diagram', 'make flowchart', 'make illustration', 'make graphic', 'make artwork',
#     'render image', 'render picture', 'render photo', 'render figure', 'render diagram', 'render flowchart', 'render illustration', 'render graphic', 'render artwork',
#     'produce image', 'produce picture', 'produce photo', 'produce figure', 'produce diagram', 'produce flowchart', 'produce illustration', 'produce graphic', 'produce artwork',
#     'design image', 'design picture', 'design photo', 'design figure', 'design diagram', 'design flowchart', 'design illustration', 'design graphic', 'design artwork',
#     'sketch image', 'sketch picture', 'sketch photo', 'sketch figure', 'sketch diagram', 'sketch flowchart', 'sketch illustration', 'sketch graphic', 'sketch artwork'
# ]
# max_skip_words = 3
# rating_labels = [1, 2, 3, 4, 5]
# stemmer = PorterStemmer()

# def stem_text(text):
#     return [stemmer.stem(word) for word in re.findall(r'\b\w+\b', str(text).lower())]

# def matches_term(review_words, term_words, max_skip_words):
#     first_word, second_word = term_words
#     first_positions = [index for index, word in enumerate(review_words) if word == first_word]
#     second_positions = [index for index, word in enumerate(review_words) if word == second_word]
#     return any(0 < abs(first_position - second_position) <= max_skip_words + 1 for first_position in first_positions for second_position in second_positions)

# stemmed_terms = [stem_text(term) for term in search_terms]

# analysis_df = df_ChatGPT[['review_text', 'review_rating_original', 'review_rating', 'sentiment']].copy()
# analysis_df['sentiment'] = pd.to_numeric(analysis_df['sentiment'], errors='coerce')
# analysis_df['review_rating_original'] = pd.to_numeric(analysis_df['review_rating_original'], errors='coerce')
# analysis_df['review_rating'] = pd.to_numeric(analysis_df['review_rating'], errors='coerce')
# analysis_df = analysis_df.dropna(subset=['review_rating_original', 'review_rating', 'sentiment'])

# stemmed_reviews = analysis_df['review_text'].fillna('').apply(stem_text)
# term_mask = stemmed_reviews.apply(lambda review_words: any(matches_term(review_words, term_words, max_skip_words) for term_words in stemmed_terms))
# matched_reviews = analysis_df[term_mask].copy()
# matched_stemmed_reviews = stemmed_reviews[term_mask]

# original_summary = matched_reviews.groupby('review_rating_original')['sentiment'].agg(original_rating_matches='count', original_rating_sentiment_score='mean').reindex(rating_labels)
# proposed_summary = matched_reviews.groupby('review_rating')['sentiment'].agg(proposed_sentiment_label_matches='count', proposed_sentiment_label_score='mean').reindex(rating_labels)
# term_summary = pd.concat([original_summary, proposed_summary], axis=1)

# match_columns = ['original_rating_matches', 'proposed_sentiment_label_matches']
# score_columns = ['original_rating_sentiment_score', 'proposed_sentiment_label_score']

# term_summary[match_columns] = term_summary[match_columns].fillna(0).astype(int)

# original_scores = term_summary['original_rating_sentiment_score']
# proposed_scores = term_summary['proposed_sentiment_label_score']

# original_average_gap = original_scores.diff().iloc[1:].mean(skipna=False)
# proposed_average_gap = proposed_scores.diff().iloc[1:].mean(skipna=False)

# percentage_improvement = (
#     (proposed_average_gap - original_average_gap) / abs(original_average_gap) * 100
#     if pd.notna(original_average_gap) and original_average_gap != 0
#     else float('nan')
# )

# term_summary[score_columns] = term_summary[score_columns].round(4)
# term_summary.index.name = 'label'

# term_counts = {
#     term: int(matched_stemmed_reviews.apply(lambda review_words: matches_term(review_words, term_words, max_skip_words)).sum())
#     for term, term_words in zip(search_terms, stemmed_terms)
# }

# top_terms = (
#     pd.Series(term_counts, name='matching_reviews')
#     .sort_values(ascending=False)
#     .head(5)
#     .to_frame()
# )
# top_terms['share_of_matches (%)'] = (100 * top_terms['matching_reviews'] / len(matched_reviews)).round(2)
# top_terms.index.name = 'search_term'

# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', 1000)

# print(f'Total matching reviews: {len(matched_reviews):,}')

# print(f'\nOriginal ratings, average sentiment gap between adjacent labels: {original_average_gap:.4f}')
# print(f'Proposed labels, average sentiment gap between adjacent labels: {proposed_average_gap:.4f}')
# print(f'Percentage improvement in the average sentiment gap between adjacent labels: {percentage_improvement:+.2f}%')

# print(f'\n{term_summary.to_string(line_width=1000)}')

# print('\nTop five most frequent search terms in the matching reviews:')
# print(top_terms.to_string(line_width=1000))

Total matching reviews: 101

Original ratings, average sentiment gap between adjacent labels: 0.1183
Proposed labels, average sentiment gap between adjacent labels: 0.2129
Percentage improvement in the average sentiment gap between adjacent labels: +79.95%

       original_rating_matches  original_rating_sentiment_score  proposed_sentiment_label_matches  proposed_sentiment_label_score
label                                                                                                                            
1                           15                           0.1121                                29                          0.0139
2                            4                           0.0233                                18                          0.0722
3                           14                           0.1746                                 7                          0.3610
4                           13                           0.2664                             